# Chapter 4: Baker's Method & Continued Fraction Reduction (General Cases)

**Context:**
This script executes the Baker's Method and continued fraction reduction to completely solve the Thue equations $F_t(X,Y) = \mu$ for all general cases where the Galois group $\mathcal{G}_t \not\simeq D_4$. The roots here are multiplicatively independent, allowing for the application of the theorem of Laurent.

**Methodology:**
1. **Laurent's Theorem:** Applies a theorem of Laurent to a real linear form in two logarithms. We obtain from this the initial upper bound of $\log|y| < 7.5 \times 10^7$ (used in the proof of Lemma 4.2.1).
2. **Legendre's Criterion (CF Reduction):** Given the initial bound on $|y|$, the script applies Legendre's theorem to the continued fraction convergents of $\log|\alpha'|/\log|\alpha|$. This allows us to lower the upper bound on $|y|$ down to $< 40$.
3. **Direct Verification:** For the four $t$ values that are not eliminated (in the case $j=3$), any potential $(x,y)$ are directly evaluated.

**Key Functions:**
* `verify_height_bounds()`: Computes the absolute logarithmic height bounds.
* `find_y_upper_bound()`: Establishes the initial $\log|y|$ upper bound.
* `execute_cf_reductions()`: Performs the continued fraction reduction and directly checks possible $(x,y)$ for any uneliminated $t$.

* *Note: 200-bit precision is used for the continued fraction reduction.*

In [ ]:
import itertools
from itertools import product

# =============================================================================
# GLOBAL SETTINGS
# =============================================================================
RUN_EXHAUSTIVE_SEARCH = True

# 200-bit precision for CF reductions.
RR_val = RealField(200)
CC_val = ComplexField(200)

# =============================================================================
# SHARED HELPER FUNCTIONS
# =============================================================================

def get_imaginary_quad_ints(M):
    """Returns a dictionary of valid imaginary quadratic integers up to bound M."""
    dlist = []
    for d in range(1, floor(4 * M**2 - 1) + 1):
        rem = d % 4
        if (rem in (1, 2) and d <= M**2) or rem == 3:
            if Integer(d).is_squarefree():
                dlist.append(d)
                
    big_set_of_integers = {}
    x_qq = PolynomialRing(QQ, 'x').gen()
    
    for d in dlist:
        ablist = set()
        L = NumberField(x_qq**2 + d, 'rootd')
        rootd = L.gen()
        for i in range(0, floor(M**2) + 1):
            elts = L.elements_of_norm(i)
            if Integer(i).is_square():
                iroot = L(sqrt(i))
                for u in L.roots_of_unity():
                    elts.append(iroot * u)
            for j in elts:
                a = QQ(j.trace() / 2)
                b = QQ((j - a) / rootd)
                if a != 0 and b != 0 and not (d == 14 and a**2 == 9 and b**2 == 1): # Exclude D4
                    ablist.add((abs(a), abs(b)))
        big_set_of_integers[d] = ablist
    return dlist, big_set_of_integers

def generate_valid_t_data(m_bound):
    """Generates pre-computed roots and derivatives for valid t."""
    dlist, all_ints = get_imaginary_quad_ints(m_bound)
    
    x_qq = PolynomialRing(QQ, 'x').gen()
    CC_poly = PolynomialRing(CC_val, 'x')
    x_cc = CC_poly.gen()
    
    for d in dlist:
        for a, b in all_ints[d]:
            abst = RR_val(sqrt(a**2 + d*b**2))
            
            # Check irreducibility over Number Field L_t
            dpol = x_qq**2 - 2 * a * x_qq + a**2 + d * b**2
            dpoldiscrim = Integer((2 * a)**2 - 4 * (a**2 + d * b**2))
            
            if dpoldiscrim.is_square():
                continue
                
            L = NumberField(dpol, 't')
            t_nf = L.gen()
            R = PolynomialRing(L, 'y')
            y = R.gen()
            
            f = y**2 - t_nf * y - 4
            ft = y**4 - t_nf * y**3 - 6 * y**2 + t_nf * y + 1
            
            if not f.is_irreducible() or not ft.is_irreducible():
                continue
            
            # Complex approximations
            tt = CC_val(a + sqrt(-d)*b)
            ftt = x_cc**4 - tt*x_cc**3 - 6*x_cc**2 + tt*x_cc + 1
            ftp = ftt.derivative()
            
            roots = [CC_val(r[0]) for r in ftt.roots()]
            alpha0 = min(roots, key=abs)
            a3 = CC_val(-(alpha0 + 1)/(alpha0 - 1))
            
            yield d, a, b, abst, alpha0, a3, ftp, roots


# =============================================================================
# PART 1: Absolute Logarithmic Height Bounds
# =============================================================================

def verify_height_bounds(m_bound):
    """Finds the smallest real number r so that h(alpha) < 1/4 * log|t| + r."""
    max_r = 0.0
    for d, a, b, abst, alpha0, _, _, roots in generate_valid_t_data(m_bound):
        abslogroots = [abs(log(abs(root))) for root in roots]
        heightbd = (1.0/8.0) * sum(abslogroots)
        r = float(heightbd - (1.0/4.0) * log(float(abst)))
        if r > max_r:
            max_r = r
    return max_r


# =============================================================================
# PART 2: Laurent's Theorem Bounds on |y|
# =============================================================================

def get_laurent_lower_bound_fast(log_y, log_t, varrho, varmu):
    """Obtain the constants for Laurent's Theorem."""
    a1 = (varrho + 9.0)*log_t + (varrho + 1.0)*1.76 + 14.08
    a2 = a1 
    varsigma = (1.0 + 2.0*varmu - varmu**2) / 2.0
    varlambda = varsigma * math.log(varrho)
    
    Iabst = 16.0 * log_t**2 + 24.16
    loglog_y = math.log(log_y)
    part1_expr = 16.0 * (loglog_y + math.log(9.02 * Iabst + 10.3) - math.log(a1) + math.log(varlambda) + 1.75) + 0.06
    
    H = part1_expr / varlambda + 1.0 / varsigma
    varomega = 2.0 * (1.0 + math.sqrt(1.0 + 1.0/(4.0*H**2)))
    vartheta = math.sqrt(1.0 + 1.0/(4.0*H**2)) + 1.0/(2.0*H)
    
    part2a = 8.0 * varlambda * varomega**(1.25) * vartheta**(0.25)
    part2b = 3.0 * math.sqrt(a1 * a2) * math.sqrt(H)
    part3 = (4.0/3.0) * (1.0/a1 + 1.0/a2) * (varlambda * varomega) / H
    
    C = (varmu / (varlambda**3 * varsigma)) * (varomega/6.0 + 0.5 * math.sqrt(varomega**2/9.0 + part2a/part2b + part3))**2
    Cprime = math.sqrt(C * varsigma * varomega * vartheta / (varlambda**3 * varmu))
    
    part0 = (part1_expr + varlambda / varsigma)
    Lambda_lower_bound = -C * part0**2 * a1 * a2 - math.sqrt(varomega * varlambda) * part0 - math.log(Cprime * part0**2 * a1 * a2)
    
    return Lambda_lower_bound

def find_y_upper_bound(abstL_sq, abstU_sq, varrho, varmu):
    """Iterates through |t|^2 to find the maximum bound on log|y|."""
    logmaxY = math.log(40.0)
    
    for normt in range(int(abstL_sq), int(abstU_sq) + 1):
        abst = math.sqrt(normt)
        log_t = 0.5 * math.log(normt)
        Iabst = 16.0 * log_t**2 + 24.16
        
        if normt < 100:
            def lambda_upper(logy): return math.log(Iabst * 130.06) - 4.0 * logy - log_t
        else:
            def lambda_upper(logy): return math.log(Iabst * 70.47) - 4.0 * logy - log_t
            
        logabsy = logmaxY
        L_val, U_val = -10.0**10, 10.0**10
        
        while L_val <= U_val:
            L_val = get_laurent_lower_bound_fast(logabsy, log_t, varrho, varmu)
            U_val = lambda_upper(logabsy)
            logabsy *= 1.01
            
        final_y_bound = logabsy / 1.01
        if final_y_bound > logmaxY:
            logmaxY = final_y_bound
            
    return logmaxY


# =============================================================================
# PART 3: Continued Fraction Reduction
# =============================================================================

def calculate_Bt_Ct(derivativeval, diff1, diff2, abst):
    Bt = 8 / (derivativeval / abst)
    z1num, z2num = Bt / diff1, Bt / diff2
    z1bd, z2bd = z1num / (40**4 * abst), z2num / (40**4 * abst)
    Ct = (z1num / (1 - z1bd)) + (z2num / (1 - z2bd))
    return ceil(1000 * Bt) / 1000, ceil(1000 * Ct) / 1000

def get_fib_index(N):
    i = 0
    while fibonacci(i) < N: i += 1
    return i

def new_maxdenom(log_yval, abst, j):
    base = (4 / 0.254) * (math.log(float(abst))**2 + 1.51)
    if j == 0: return float(base * (2 * (1.78 * float(log_yval) + 1.48) + 1))
    if j == 3: return float(base * (2 * (1.78 * float(log_yval) + 2.64)))

def find_min_y(ymax_bound, abst, absalpha, Bt):
    var('x')
    beta_fn = absalpha * x**4 - 40 * x**3 + Bt / abst
    beta_fn_min_x = float(120 / (4 * absalpha))
    return find_root(beta_fn, beta_fn_min_x, float(ymax_bound))

def iterative_cf_test(cf, idx_bound, maxdenom, diffs, cst, Ibd, logalpha, abst):
    max_yval = 0
    for i in range(1, idx_bound):
        pq = cf.convergent(i)
        q = pq.denominator()
        if q > maxdenom: break
            
        RHS = float(diffs[i])
        if RHS == 0: continue
            
        LHS = float(cst * Ibd) / (float(logalpha) * float(abst) * float(q))
        
        root = (LHS / RHS)**0.25
        yval = max(1, int(root) - 1)
        while float(LHS / (yval**4)) > RHS: 
            yval += 1
            
        if yval > max_yval:
            max_yval = yval
            
    return max_yval

def final_j3_elim(alpha3, alpha13diff, ymin, ymax, abst, dval, Bt):
    absalpha3 = abs(alpha3)
    x2set = set()
    Ltemp = QuadraticField(-dval, 'temproot')
    temproot = Ltemp.gen()
    
    for y2 in range(int(ymin**2), int(ymax**2) + 1):
        absy = sqrt(y2)
        error_term = RR_val(Bt / (absy**3 * abst))
        
        x2lb = max(ceil((absalpha3 * absy - error_term)**2), 40**2)
        x2ub = floor((absalpha3 * absy + error_term)**2)
        
        if x2lb <= x2ub:
            for x2 in range(x2lb, x2ub + 1):
                xy = QQ(x2 / y2)
                if xy > (alpha13diff - 1/absalpha3 - RR_val(Bt/(y2**2 * abst))):
                    continue
                if xy.is_norm(Ltemp):
                    for x_el, y_el in itertools.product(Ltemp.elements_of_norm(x2), Ltemp.elements_of_norm(y2)):
                        ratio = CC_val(x_el / y_el)
                        if abs(ratio - alpha3) < RR_val(Bt/absy) or abs(ratio + alpha3) < RR_val(Bt/absy):
                            x_tr, y_tr = x_el.trace() / 2, y_el.trace() / 2
                            x_im = ((x_el - x_tr) / temproot) * sqrt(-dval)
                            y_im = ((y_el - y_tr) / temproot) * sqrt(-dval)
                            x2set.add((x2, y2, x_tr + x_im, y_tr + y_im))
    return x2set

def execute_cf_reductions(m_bound=10):
    """Executes the complete CF reduction loop to reduce bounds on y."""
    ymax_log_start = 7.5 * 10**7
    
    for d, a, b, abst, alpha0, a3, ftp, roots in generate_valid_t_data(m_bound):
        logalpha = float(abs(log(abs(alpha0))))
        a01diff, a03diff = abs(alpha0 + 1/a3), abs(alpha0 - a3)
        a13diff, a23diff = abs(a3 + 1/a3), abs(a3 + CC_val(1/alpha0))
        
        ftpa0, ftpa3 = abs(ftp(alpha0)), abs(ftp(a3))
        Bt0, Ct0 = calculate_Bt_Ct(ftpa0, a01diff, a03diff, abst)
        Bt3, Ct3 = calculate_Bt_Ct(ftpa3, a23diff, a03diff, abst)
        
        # RR_val preserves 200 bits of precision for convergent tracking
        gamma = RR_val(log(abs(a3)) / logalpha)
        cf = continued_fraction(gamma)
        
        abst_float = float(abst)
        Ibd = (4 / 0.253) * (math.log(abst_float)**2 + 1.51) if abst_float < 10 else (4 / 0.253) * (math.log(abst_float + 0.802)**2 + math.log(1.20405)**2)
        
        # Generate initial denominators with log values
        maxdenom0, maxdenom3 = new_maxdenom(ymax_log_start, abst, 0), new_maxdenom(ymax_log_start, abst, 3)
        ylb = find_min_y(ymax_log_start, abst, float(abs(alpha0)), Bt0)
        ylb3 = 40
        
        idx_bound = get_fib_index(max(maxdenom0, maxdenom3))
        diffs = [float(abs(gamma - cf.convergent(i))) for i in range(idx_bound + 1)]
        
        yval0 = iterative_cf_test(cf, idx_bound, maxdenom0, diffs, Ct0, Ibd, logalpha, abst)
        yval3 = iterative_cf_test(cf, idx_bound, maxdenom3, diffs, Ct3, Ibd, logalpha, abst)
        
        # 2nd Pass Reduction
        maxdenom0, maxdenom3 = new_maxdenom(math.log(max(yval0, 2)), abst, 0), new_maxdenom(math.log(max(yval3, 2)), abst, 3)
        idx_bound = get_fib_index(max(maxdenom0, maxdenom3))
        if not (yval0 < ylb and yval3 < ylb3):
            yval0 = iterative_cf_test(cf, idx_bound, maxdenom0, diffs, Ct0, Ibd, logalpha, abst)
            yval3 = iterative_cf_test(cf, idx_bound, maxdenom3, diffs, Ct3, Ibd, logalpha, abst)
        
        # 3rd Pass Reduction
        maxdenom0 = new_maxdenom(math.log(max(yval0, 2)), abst, 0)
        idx_bound = get_fib_index(maxdenom0)
        if not (yval0 < ylb and yval3 < ylb3):
            yval0 = iterative_cf_test(cf, idx_bound, maxdenom0, diffs, Ct0, Ibd, logalpha, abst)
            yval3 = iterative_cf_test(cf, idx_bound, maxdenom3, diffs, Ct3, Ibd, logalpha, abst)
            
        if yval0 >= ylb:
            print(f"t = {a}+sqrt({-d})*{b}, |y|<{yval0}, |y|>{ylb}, j=0")
        if yval3 >= ylb3:
            xyset = final_j3_elim(a3, a13diff, ylb3, yval3, abst, d, Bt3)
            if xyset:
                # Direct check step: evaluate F_t(px, py) for each survivor to eliminate false positives
                tt = CC_val(a + sqrt(-d)*b)
                true_survivors = set()
                for x2, y2, px, py in xyset:
                    F_val = abs(px**4 - tt * py * px**3 - 6 * px**2 * py**2 + tt * px * py**3 + py**4)
                    if float(F_val) <= 2.0:
                        true_survivors.add((x2, y2, px, py))
                        
                if true_survivors:
                    print(f"t = {a}+sqrt({-d})*{b}, j=3, Survivors: {true_survivors}")


# =============================================================================
# MAIN RUNNER
# =============================================================================

if __name__ == "__main__":
    if RUN_EXHAUSTIVE_SEARCH:
        print("=== Lemma 4.1: Height Bounds ===")
        max_r = verify_height_bounds(10)
        print(f"  • h(|alpha|) < 1/4 * log|t| + {max_r:.4f}")
        
        print("\n=== Lemma 4.2.1: Laurent's Theorem log|y| Bound ===")
        y_bound = find_y_upper_bound(1, 10000, varrho=20.0, varmu=1.0/3.0)
        print(f"  • log|y| < {y_bound:.4f} for 1 <= |t|^2 <= 10000")
        
        print("\n=== Lemma 4.2.3: Continued Fraction Reduction ===")
        print("Executing iterative reductions...")
        execute_cf_reductions(m_bound=50)
        print("Reduction complete. If no solutions printed above, none exist > 40.")
    else:
        print("Search disabled. Set RUN_EXHAUSTIVE_SEARCH = True to execute.")

Search disabled. Set RUN_EXHAUSTIVE_SEARCH = True to execute.
